# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abc085455-byte/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# --- Setup: make this notebook work whether it's opened locally or via the Colab badge ---
import os, pathlib, subprocess

REPO_URL = "https://github.com/abc085455-byte/flyrank-ml-internship.git"

def find_repo_root(start="."):
    p = pathlib.Path(start).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "skills" / "README.md").exists() and (candidate / "docs").exists():
            return candidate
    return None

repo_root = find_repo_root()
if repo_root is None:
    clone_dir = pathlib.Path("/content/flyrank-ml-internship")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    repo_root = clone_dir

os.chdir(repo_root)
print("Working directory set to:", os.getcwd())


Working directory set to: /content/flyrank-ml-internship


## 1. Question

*The research question and the decision it supports.*

**Question:** Given a portfolio of already-published content pages, which pages should a
content/SEO reviewer look at first this week — and why — when they only have time to review a
small number?

**Decision this supports:** a content editor (or their team lead) triaging a backlog picks the
top of a ranked queue instead of scanning thousands of pages in publish order or client order.
This is the same decision framed in `w01_research_question.ipynb`: reviewing a page that didn't
need it costs 1-3 hours of wasted labor (a false positive); missing a real decliner lets it keep
losing visibility for weeks before anyone notices (a false negative, the costlier mistake). The
scoring below is built to put real decliners near the top of the queue, not just to maximize
overall accuracy.

**Lane:** Refresh / Content Opportunity Scoring — a scoring task (a probability per page, used to
rank a queue), built on a binary classification sub-task (`is_declining_label`), per
`w02_ml_task_framing.ipynb`.


In [2]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df):,} | Clients: {df['client_id'].nunique()} | Content types: {df['content_type'].nunique()}")
print(f"Share matching is_declining_label (proxy: trend_direction == 'down'): {df['is_declining_label'].mean():.3f}")
print(f"Median impressions_90d (eligible population has real search demand): {df['impressions_90d'].median():.0f}")


Rows: 30,000 | Clients: 32 | Content types: 3
Share matching is_declining_label (proxy: trend_direction == 'down'): 0.542
Median impressions_90d (eligible population has real search demand): 731


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release used:** the starter dataset shipped in this repo,
`data/raw/content_refresh_anonymized.csv` — 30,000 pseudonymized content items, 44 columns, 32
pseudonymized clients, one row per content item with trailing-90-day rolled-up metrics (per
`docs/data-dictionary.md` and `skills/flyrank/flyrank-data/SKILL.md`). This is a single snapshot,
not a repeated time series per page — there is no per-client publish calendar in this slice, so a
true time-based split isn't available here (see Methodology).

**Not used:** the larger Hugging Face warehouse release (`fact_content_daily_performance`,
~79M rows) was scoped out for this capstone. It would enable a genuine future-window label (train
on one period, predict the next) instead of the current-window proxy used here, but its per-client
history depth is uneven and several of its tables carry window-overlap risk that needs careful
alignment (`skills/flyrank/flyrank-data/SKILL.md`) — a good next step, named honestly in
Limitations, not attempted under this project's time budget.

**Excluded columns, and why** (full detail in `w03_feature_leakage_check.ipynb`):
- `trend_direction`, `trend_pct` — these define the label itself; using them as features would be
  circular (`skills/flyrank/flyrank-data/SKILL.md`'s "label trap").
- `content_id`, `client_id` — pseudonyms, used only to group the train/test split, never as
  model inputs.
- No FlyRank product flags (`health_score`, `priority_score`, `is_quick_win`, etc.) exist in this
  starter file at all — confirmed by a column-name scan in `w06_validation_audit.ipynb`.

**Public-safety:** every field in this dataset is already pseudonymized by FlyRank before release
(no client names, no raw query text, no URLs) — nothing in this paper or its source notebooks
adds any real-world identifying information back in.


In [3]:
import json

data_summary = {
    "dataset": "data/raw/content_refresh_anonymized.csv (starter release)",
    "rows": int(len(df)),
    "clients": int(df["client_id"].nunique()),
    "content_types": sorted(df["content_type"].dropna().unique().tolist()),
    "columns": int(df.shape[1]),
    "time_window": "single trailing-90-day snapshot per row (no repeated per-client timestamps)",
    "excluded_as_label_source": ["trend_direction", "trend_pct"],
    "excluded_as_ids": ["content_id", "client_id"],
    "warehouse_release_used": False,
}
for k, v in data_summary.items():
    print(f"{k}: {v}")


dataset: data/raw/content_refresh_anonymized.csv (starter release)
rows: 30000
clients: 32
content_types: ['comparison article', 'feedly article', 'keyword article']
columns: 45
time_window: single trailing-90-day snapshot per row (no repeated per-client timestamps)
excluded_as_label_source: ['trend_direction', 'trend_pct']
excluded_as_ids: ['content_id', 'client_id']
warehouse_release_used: False


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining_label = (trend_direction == "down")` — a **current-window proxy**, not a
future outcome. `trend_direction` itself summarizes the last 30 days vs. the prior 30 days of
impressions (per `docs/data-dictionary.md`). This means the model answers "is this page
*currently* trending down," not "will it decline *next*" — named as a limit, not hidden (Section 5).

**Features (18 numeric + 8 one-hot categorical, 53 columns after encoding):** log-transformed
volume metrics (`impressions_90d`, `clicks_90d`, `sessions_90d`, `ai_sessions_90d`), engagement and
CTR metrics, position, content age and freshness, word count, keyword-difficulty fields, and
`has_search_volume` / `has_word_count` missingness flags (added per the data dictionary's warning
against blind `fillna(0)` on fields where missingness follows `content_type`).

**Baseline (`w04_baseline_score.ipynb`):** a transparent rule, not a fitted model — flag a page
when it's visible (`impressions_90d >= 500`, `avg_position > 0`) **and** its CTR sits below the
median CTR of other visible pages in its own position tier; score by `impressions_90d`. Built
after confirming CTR genuinely tracks position tier in this data (Signal B, `w04_signal_audit`
lane), and after finding raw staleness alone does *not* reliably predict decline here (Signal A,
verdict **MIXED** — a `181+`-day-stale bucket showed a *lower* decline rate than average, so
staleness was deliberately left out of the rule).

**Models:** Logistic Regression, then Random Forest (`w05_model.ipynb`) — starting simple and
readable before reaching for a stronger, less transparent model, per this track's
`training-honest-models` skill.

**Validation design — the honest split:** `GroupShuffleSplit` on `client_id`, 80/20,
`random_state=42` — not a random row split, because `client_id` groups pages that share a brand,
CMS, and editorial team. A **naive random split leaves 31 of the 32 clients present on both sides
of train/test**; the client-grouped split leaves **0 client overlap** (25 train clients / 7 held-out
test clients). The gap this exposes is reported in Results, not hidden.

**Leakage checks (`w06_validation_audit.ipynb`):** (1) injecting `trend_pct` (the label's own
numerator) lifts Logistic Regression's ROC AUC from 0.615 to 0.999, and injecting a literal copy
of the label hits 1.000 — confirming the test harness is sensitive enough to trust, and confirming
neither leaky column is in the real feature set; (2) removing `days_with_impressions` (the top
permutation-importance feature) barely moves Logistic Regression's AUC (0.615 -> 0.615) and costs
Random Forest only ~0.01-0.02 AUC — a real feature, not a disguised label; (3) a column-name scan
found no product-flag features to accidentally reuse; (4) the 90-day feature window technically
overlaps the label's own 30-day sub-windows, which is expected for this same-period triage task
(not a forward-looking forecast) and is named as a scope limit, not a leak, since the correlation
between `impressions_90d` and the label is weak (0.18) — nowhere near the ~1.0 a disguised copy
would show.


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score, f1_score,
)

RANDOM_STATE = 42

numeric_features = [
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
    "word_count", "has_search_volume", "has_word_count",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_search_volume"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

num = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[categorical_features].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].to_numpy()
groups = df["client_id"].to_numpy()

# The honest, client-grouped split -- computed here so the baseline below can be fit on
# train only, exactly like the models, instead of peeking at test rows.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

# --- baseline rule, same logic as w04_baseline_score.ipynb, fit on TRAIN ONLY ---
# (fair comparison: the position-tier median CTR threshold is learned from training clients
# only, then applied to the held-out test clients -- same discipline as the fitted models,
# even though a median is a robust statistic that barely moves either way here.)
visible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
train_mask = np.zeros(len(df), dtype=bool)
train_mask[train_idx] = True
tier_median_ctr_train = df.loc[visible & train_mask].groupby("position_tier")["ctr"].median()
tier_median_applied = df["position_tier"].map(tier_median_ctr_train)

baseline_flag = pd.Series(0, index=df.index)
baseline_flag.loc[visible] = (df.loc[visible, "ctr"] < tier_median_applied.loc[visible]).astype(int)
baseline_score = pd.Series(np.where(baseline_flag == 1, df["impressions_90d"], 0), index=df.index)

print(f"Feature matrix: {X.shape}, label base rate: {y.mean():.3f}")
print(f"Baseline flags {int(baseline_flag.sum())} of {len(df)} rows ({baseline_flag.mean():.1%})")
print(f"Test clients held out: {df.iloc[test_idx]['client_id'].nunique()} / test base rate: {y[test_idx].mean():.3f}")


Feature matrix: (30000, 53), label base rate: 0.542
Baseline flags 7966 of 30000 rows (26.6%)
Test clients held out: 7 / test base rate: 0.511


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

All three (baseline rule, Logistic Regression, Random Forest) are scored on the **same
client-grouped test split** — 7 held-out clients the models never trained on, a test-set base
rate of 0.511. Headline metric is **precision@50** (of the top 50 ranked pages, how many are
really declining) — chosen in `w02_ml_task_framing.ipynb` because the real action is "a human
reviews a short, capacity-limited list," not "rank all 30,000 pages perfectly."


In [5]:
def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": y_true, "score": scores})
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean())

results = {}

# baseline, same split, thresholds fit on train only (see cell above)
base_scores_test = baseline_score.iloc[test_idx].to_numpy()
base_flag_test = baseline_flag.iloc[test_idx].to_numpy()
results["baseline_rule"] = {
    "roc_auc": roc_auc_score(y[test_idx], base_scores_test),
    "precision_at_50": precision_at_k(y[test_idx], base_scores_test, 50),
    "precision@flag": precision_score(y[test_idx], base_flag_test, zero_division=0),
    "recall@flag": recall_score(y[test_idx], base_flag_test, zero_division=0),
    "f1@flag": f1_score(y[test_idx], base_flag_test, zero_division=0),
}

log_reg = Pipeline([("scaler", StandardScaler()),
                     ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
log_reg.fit(X.iloc[train_idx], y[train_idx])
lr_proba = log_reg.predict_proba(X.iloc[test_idx])[:, 1]
results["logistic_regression"] = {
    "roc_auc": roc_auc_score(y[test_idx], lr_proba),
    "precision_at_50": precision_at_k(y[test_idx], lr_proba, 50),
    "precision@flag": precision_score(y[test_idx], (lr_proba >= 0.5).astype(int), zero_division=0),
    "recall@flag": recall_score(y[test_idx], (lr_proba >= 0.5).astype(int), zero_division=0),
    "f1@flag": f1_score(y[test_idx], (lr_proba >= 0.5).astype(int), zero_division=0),
}

rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                             n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X.iloc[train_idx], y[train_idx])
rf_proba = rf.predict_proba(X.iloc[test_idx])[:, 1]
results["random_forest"] = {
    "roc_auc": roc_auc_score(y[test_idx], rf_proba),
    "precision_at_50": precision_at_k(y[test_idx], rf_proba, 50),
    "precision@flag": precision_score(y[test_idx], (rf_proba >= 0.5).astype(int), zero_division=0),
    "recall@flag": recall_score(y[test_idx], (rf_proba >= 0.5).astype(int), zero_division=0),
    "f1@flag": f1_score(y[test_idx], (rf_proba >= 0.5).astype(int), zero_division=0),
}

results_table = pd.DataFrame(results).T.round(3)
print(f"Test-set base rate: {y[test_idx].mean():.3f} | test clients: {df.iloc[test_idx]['client_id'].nunique()}")
print(results_table.to_string())


Test-set base rate: 0.511 | test clients: 7
                     roc_auc  precision_at_50  precision@flag  recall@flag  f1@flag
baseline_rule          0.538             0.50           0.600        0.271    0.373
logistic_regression    0.615             0.72           0.588        0.622    0.605
random_forest          0.607             0.64           0.588        0.598    0.593


**Read plainly — and a claim I corrected while building this.** An earlier draft of this
paper (following the numbers already floating in `w01_research_question.ipynb` /
`w02_ml_task_framing.ipynb`, precision@50 0.24 for the rule vs. 0.74 for a model) would have
called this a 3x lift. Recomputing the baseline honestly, on the *same* client-grouped split as
the models — thresholds fit on training clients only, evaluated on the 7 held-out ones — the
baseline actually scores **precision@50 = 0.50**, not 0.24. The 0.24 figure came from the
reference pipeline's naive random split; it was never re-evaluated on the honest split before
being repeated in earlier weeks. Recomputed fairly, Logistic Regression's precision@50 (0.72) is
a real but more modest **~1.4x** improvement over the baseline (0.50) on the exact same held-out
clients — 36 of the top 50 flagged pages are really declining, versus about 25 of 50 for the
rule. Random Forest scores close behind (0.64). Logistic Regression is the model this paper's
playbook uses (Section 6), consistent with `w06_validation_audit.ipynb`'s finding that it beat
Random Forest under the honest split even though a naive random-split evaluation would have
favored Random Forest instead (0.94 vs 0.92) — a reminder that *which* split you validate on can
flip which model looks best, and that a number carried over from an earlier week without
re-checking its split is a claim, not a fact, until it's been rerun.

**Split-variance, stated honestly:** across five different client-grouped reseeds of this same
32-client dataset, Logistic Regression's precision@50 ranged from **0.52 to 0.72** (mean 0.64) —
so read 0.72 as one honest draw from a real range, not a fixed constant this exact model will
always hit on a new client roster.


In [6]:
seed_p50 = {}
for seed in [42, 1, 7, 13, 99]:
    gss_s = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_s, te_s = next(gss_s.split(X, y, groups=groups))
    lr_s = Pipeline([("scaler", StandardScaler()),
                      ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
    lr_s.fit(X.iloc[tr_s], y[tr_s])
    p_s = lr_s.predict_proba(X.iloc[te_s])[:, 1]
    seed_p50[seed] = precision_at_k(y[te_s], p_s, 50)

print("precision@50 (Logistic Regression, client-grouped) across 5 splits:")
for seed, p in seed_p50.items():
    print(f"  seed {seed:3d}: {p:.2f}")
print(f"  min={min(seed_p50.values()):.2f}, max={max(seed_p50.values()):.2f}, mean={np.mean(list(seed_p50.values())):.2f}")


precision@50 (Logistic Regression, client-grouped) across 5 splits:
  seed  42: 0.72
  seed   1: 0.66
  seed   7: 0.62
  seed  13: 0.52
  seed  99: 0.70
  min=0.52, max=0.72, mean=0.64


## 5. Limitations

*What this work cannot claim.*

- **Current-window label, not a future forecast.** `is_declining_label` captures the last 30 days
  vs. the prior 30, not chronic multi-quarter decline. A page that's been flat-bad for months
  (not newly getting worse) can read as a false alarm here even though it may still deserve a
  human's attention (`w05_model.ipynb`, error case #3).
- **No causal claim.** The paper's own reference finding — that older, refreshed pages show a
  large gap over older, un-refreshed pages — is an association, not an experiment: which pages
  get refreshed is a human choice, not random, so part of any observed gap could be "teams already
  pick pages they believe in," not "refreshing caused the lift" (audited in
  `w06_validation_audit.ipynb`, Section 1).
- **Small, fixed client pool.** Only 32 clients exist in this starter slice, so client-grouped
  validation has just 7 held-out clients per split — precision@50 ranged 0.52-0.72 across five
  reasonable splits of the same data (Section 4). Treat a new client's queue as low-confidence
  until re-validated on that client's own realized outcomes.
- **Scope.** Trained and validated on this repo's 30K-row, 32-client anonymized sample only — not
  shown to generalize to clients, industries, or content types outside it. The larger warehouse
  release (Section 2) was not used here.
- **Single snapshot, not a live feed.** This is one trailing-90-day cut, not a continuously
  updated pipeline — scores go stale as new performance data arrives (monitoring cadence in
  Section 6).
- **The model sees numbers only.** No title, body text, brand voice, or images enter the feature
  set — a "low CTR" flag could mean a bad title or an oddly-matched query, and only a human
  reading the page can tell which (`w07_action_playbook.ipynb`, human-review rules).


In [7]:
print("Limitations, made checkable:")
print(f"  training population size: {len(df):,} rows, {df['client_id'].nunique()} clients")
print(f"  label window: last-30d vs prior-30d impressions (a current-window proxy, not a forecast)")
print(f"  precision@50 observed range across 5 client-grouped splits: "
      f"{min(seed_p50.values()):.2f}-{max(seed_p50.values()):.2f}")
print(f"  warehouse release (79M rows) used in this capstone: False")


Limitations, made checkable:
  training population size: 30,000 rows, 32 clients
  label window: last-30d vs prior-30d impressions (a current-window proxy, not a forecast)
  precision@50 observed range across 5 client-grouped splits: 0.52-0.72
  warehouse release (79M rows) used in this capstone: False


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Full playbook: `w07_action_playbook.ipynb`. Reusing the same honest, client-grouped Logistic
Regression from Section 4, scored on the 7 held-out clients as a stand-in for "a brand-new
client's queue," with transparent, threshold-based reason codes layered on top of the model
score (not the score alone).

**Archetype -> action mapping:**

| Archetype (reason codes) | Action | Why a human trusts it |
|---|---|---|
| `sparse_data` | `collect_more_data` | Fewer than 20 impressions/90d means almost every feature is near-zero — not enough signal to say "refresh" or "leave it." |
| `thin_visible_page` | `expand_and_refresh` | Real traffic, but under 1,200 words — the gap looks like depth, not just staleness. |
| `low_ctr_visible_page` | `refresh_and_review_ctr` | Good position, weak clicks — likely a title/snippet problem. |
| `low_engagement_visible_page` + `declining_with_demand` | `refresh_and_review_engagement` | Traffic arrives but doesn't stick, and the trend is down — a body-content problem. |
| `stale_visible_page` / `declining_with_demand` / `page_one_decay_risk` | `refresh` | Visible, aging or trending down, no single sharper cause. |
| none of the above | `monitor` | Nothing urgent flagged; check back next cycle. |

**Decision-support framing, not a guarantee:** treat "refresh" as a prioritized reviewer
suggestion backed by an observed association (Section 5), not a promised lift.

**Human review, always:** read the actual page; check `confidence` alongside `reason_codes`, not
the score alone; check for context the model can't see (seasonality, a recent redesign, an
intentional sunset). **Never automated on the model's say-so alone:** publishing/editing,
depublishing/deindexing, client-facing claims ("your content is declining"), or budget/headcount
decisions.

**Monitoring / retrain triggers:** realized-outcome drift (acted-on pages' next-90-day trend
falling well below the observed 0.52-0.72 range), base-rate drift from the current 54.2% decline
rate, growth in the client roster (which should narrow the current split-variance range), and
feature-distribution drift on `log_impressions_90d` / `avg_position`. Recompute the queue every
reporting cycle regardless, since a 90-day-window model scored against month-old data is already
looking at a different reality than the one it was fit on.


In [8]:
action_mix = pd.Series({
    "refresh_and_review_ctr": 1827, "monitor": 1555, "refresh": 1426,
    "collect_more_data": 1298, "refresh_and_review_engagement": 57,
})
print("Action mix on the held-out queue (from w07_action_playbook.ipynb, 6,163 rows / 7 clients):")
print(action_mix)
print(f"\nsparse_data rows routed to collect_more_data (never auto-acted on): 1,298 of 6,163 ({1298/6163:.1%})")


Action mix on the held-out queue (from w07_action_playbook.ipynb, 6,163 rows / 7 clients):
refresh_and_review_ctr           1827
monitor                          1555
refresh                          1426
collect_more_data                1298
refresh_and_review_engagement      57
dtype: int64

sparse_data rows routed to collect_more_data (never auto-acted on): 1,298 of 6,163 (21.1%)


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Three figures already committed at `work/figures/` (generated by `w07_action_playbook.ipynb`):
`w07_action_mix.svg`, `w07_confidence_mix.svg`, `w07_precision_at_50_range.svg`. This section adds
the two comparison charts the paper's Results section needs — model vs. baseline, and before/after
the honest split — plus one summary table, all written to `work/figures/` for the deployed page to
embed directly (relative paths, no external hosting).


In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pathlib

FIGURES_DIR = pathlib.Path("work/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Figure A: model vs baseline, precision@50, on the same honest split
fig1, ax1 = plt.subplots(figsize=(6, 4))
names = ["Baseline rule", "Logistic Regression", "Random Forest"]
p50s = [results["baseline_rule"]["precision_at_50"], results["logistic_regression"]["precision_at_50"],
        results["random_forest"]["precision_at_50"]]
bars = ax1.bar(names, p50s, color=["#8F5B3B", "#3B6E8F", "#4C7A4C"])
ax1.axhline(y[test_idx].mean(), color="black", linestyle="--", linewidth=1, label="test-set base rate")
ax1.set_ylabel("precision@50")
ax1.set_title("Model vs. baseline -- same held-out clients")
ax1.set_ylim(0, 1)
ax1.legend()
for b, v in zip(bars, p50s):
    ax1.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}", ha="center")
fig1.tight_layout()
fig1.savefig(FIGURES_DIR / "capstone_model_vs_baseline.svg")
plt.close(fig1)

# Figure B: before/after the honest split (naive random vs client-grouped), same model
tr_r, te_r = train_test_split(np.arange(len(X)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)
lr_naive = Pipeline([("scaler", StandardScaler()),
                      ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
lr_naive.fit(X.iloc[tr_r], y[tr_r])
naive_p50 = precision_at_k(y[te_r], lr_naive.predict_proba(X.iloc[te_r])[:, 1], 50)
grouped_p50 = results["logistic_regression"]["precision_at_50"]

fig2, ax2 = plt.subplots(figsize=(6, 4))
bars2 = ax2.bar(["Naive random split", "Honest client-grouped split"], [naive_p50, grouped_p50],
                 color=["#B0413E", "#3B6E8F"])
ax2.set_ylabel("precision@50 (Logistic Regression)")
ax2.set_title("Same model, same features -- before vs. after the honest split")
ax2.set_ylim(0, 1)
for b, v in zip(bars2, [naive_p50, grouped_p50]):
    ax2.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}", ha="center")
fig2.tight_layout()
fig2.savefig(FIGURES_DIR / "capstone_split_before_after.svg")
plt.close(fig2)

# Summary table -> JSON, the paper's Results table source of truth
capstone_metrics = {
    "test_set_base_rate": round(float(y[test_idx].mean()), 3),
    "test_clients_held_out": int(df.iloc[test_idx]["client_id"].nunique()),
    "results_table": results_table.to_dict(orient="index"),
    "precision_at_50_naive_random_split": round(float(naive_p50), 3),
    "precision_at_50_honest_grouped_split": round(float(grouped_p50), 3),
    "precision_at_50_range_across_5_seeds": [round(min(seed_p50.values()), 3), round(max(seed_p50.values()), 3)],
    "precision_at_50_mean_across_5_seeds": round(float(np.mean(list(seed_p50.values()))), 3),
    "action_mix": action_mix.to_dict(),
}
import json
import pathlib
pathlib.Path("work/outputs").mkdir(parents=True, exist_ok=True)
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(capstone_metrics, f, indent=2)

print("Wrote:")
print(" ", FIGURES_DIR / "capstone_model_vs_baseline.svg")
print(" ", FIGURES_DIR / "capstone_split_before_after.svg")
print("  work/outputs/capstone_metrics.json")
print()
print(f"naive random split precision@50: {naive_p50:.2f}")
print(f"honest grouped split precision@50: {grouped_p50:.2f}")


Wrote:
  work/figures/capstone_model_vs_baseline.svg
  work/figures/capstone_split_before_after.svg
  work/outputs/capstone_metrics.json

naive random split precision@50: 0.92
honest grouped split precision@50: 0.72


## Closing cells (ML-12): demo outline, social post, employer summary

*A 5-minute demo outline, a social-post cut, and a 3-sentence employer-facing summary of this
capstone.*

**5-minute demo outline:**
1. *(0:00-0:45)* Open with the decision: "which of thousands of content pages should an editor
   look at first this week?" Show the ranked queue's top 5 rows live.
2. *(0:45-1:30)* Show the rule baseline (`w04_baseline_score.ipynb`) — simple, transparent,
   precision@50 = 0.50 on held-out clients (recomputed honestly on the same split as the model;
   an earlier draft had mistakenly carried over a naive-split 0.24 figure from an earlier week).
3. *(1:30-2:30)* Show the honest split matters: same Logistic Regression, naive random split
   (0.92) vs. client-grouped split (0.72) — walk through *why* (client memorization), not just
   the number.
4. *(2:30-3:30)* Show the model vs. baseline chart: 0.50 -> 0.72, a real but modest ~1.4x
   lift, on the same 7 never-before-seen clients.
5. *(3:30-4:15)* Show one archetype -> action mapping row (`low_ctr_visible_page` ->
   `refresh_and_review_ctr`) and the no-go list — what a human must check, what never gets
   automated.
6. *(4:15-5:00)* Close on limitations stated up front (current-window label, no causal claim,
   32-client split variance) and the data credit.

**Social-post cut** (one finding + one chart + one method sentence + link):

> I built a content-decline scoring model on FlyRank's 30K-row SEO internship dataset. The
> headline finding: switching from a naive random train/test split to an honest, client-grouped
> split dropped my model's precision@50 from 0.92 to 0.72 — the "worse" number is the real one,
> because the first was partly memorization. Full paper + reproducible notebooks: [deployed URL].

**Employer-facing 3-sentence summary:**

> I built and honestly validated a content-prioritization model on a 30,000-row, 32-client SEO
> dataset, comparing a transparent rule baseline against Logistic Regression and Random Forest
> under a client-grouped (not randomly leaked) train/test split. The validated model lifts
> precision@50 from 0.50 (rule baseline, recomputed fairly on the same split) to 0.72 (Logistic
> Regression) on clients it never trained on — a real, modest lift, not the inflated 3x an
> unchecked earlier-week number would have implied — while I separately demonstrate and correct
> for the ~0.20 precision@50 inflation a naive random split hides on the model itself. The full
> pipeline (leakage audit, error analysis, and a human-reviewed action playbook with a no-go
> list) is reproducible end-to-end from the linked repo and notebooks.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
